# Build and inspect a DataFrame · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 04. Build and inspect a DataFrame

Turn Python data into a table and inspect its structure before analysis.

Before you start: Know dictionaries, lists, Series values, and index labels.

Official reference: https://pandas.pydata.org/docs/getting_started/intro_tutorials/01_table_oriented.html

### M04C01. Make a table from a dictionary

A DataFrame is a two-dimensional table with rows and named columns. Pass a dictionary of equally long lists to pd.DataFrame(). Each dictionary key becomes a column name. Each list position supplies the values for one row.

**Read:** What becomes the two column names?

Your explanation: _write here_.

**Predict:** What does df.shape print?

Your explanation: _write here_.

**Explain:** Does the default index count as a third data column?

Your explanation: _write here_.



In [ ]:
# Predict before running: What does df.shape print?
data = {"Name": ["Amira", "Bilal", "Chloe"], "Age": [20, 21, 19]}
df = pd.DataFrame(data)
print(df)
print(type(df).__name__)
print("Rows and columns:", df.shape)

**Your turn:** Add a Campus list with RAK, Dubai, and RAK to data before building df. Print the new table and shape.

For the check, keep these result names: `df`.

**Hint:** Add another key with a list of exactly three values.

**Watch for this:** Call pd.DataFrame(...) with a capital D and F; Python names are case-sensitive.

In [ ]:
# Change this code to complete the task.
# Question: Add a Campus list with RAK, Dubai, and RAK to data before building df. Print the new table and shape.
data = {"Name": ["Amira", "Bilal", "Chloe"], "Age": [20, 21, 19]}
df = pd.DataFrame(data)
print(df)
print(type(df).__name__)
print("Rows and columns:", df.shape)

**Quick check:** In df.shape, what comes first?

1. Column count.
2. Row count.
3. The first score.

**Explain what you learned:** I can turn a dictionary of lists into a DataFrame.

### M04C02. Build a table from records

A list of dictionaries can also become a DataFrame. Each dictionary contributes a row, and pandas matches fields by their keys. If one record lacks a field, pandas marks that cell as missing. The table therefore does not require every record to contain the same keys.

**Read:** Which record is missing the Campus key?

Your explanation: _write here_.

**Predict:** Will the second row disappear because its campus is absent?

Your explanation: _write here_.

**Explain:** Why is this different from unequal plain lists?

Your explanation: _write here_.



In [ ]:
# Predict before running: Will the second row disappear because its campus is absent?
rows = [
    {"Name": "Amira", "Age": 20, "Campus": "RAK"},
    {"Name": "Bilal", "Age": 21},
    {"Name": "Chloe", "Age": 19, "Campus": "RAK"}
]
df = pd.DataFrame(rows)
print(df)
print("Missing campus:", df["Campus"].isna().sum())

**Your turn:** Add Campus as Dubai to Bilal's record before constructing df. Check that no campus values are missing.

For the check, keep these result names: `df`.

**Hint:** Update rows[1]["Campus"] before calling pd.DataFrame(rows).

**Watch for this:** Missing-value display can vary with dtype and pandas version; inspect it with isna().

In [ ]:
# Change this code to complete the task.
# Question: Add Campus as Dubai to Bilal's record before constructing df. Check that no campus values are missing.
rows = [
    {"Name": "Amira", "Age": 20, "Campus": "RAK"},
    {"Name": "Bilal", "Age": 21},
    {"Name": "Chloe", "Age": 19, "Campus": "RAK"}
]
df = pd.DataFrame(rows)
print(df)
print("Missing campus:", df["Campus"].isna().sum())

**Quick check:** A record is missing one field. What can pd.DataFrame(rows) do?

1. Invent a likely value.
2. Delete all the other records.
3. Keep the row and mark the field as missing.

**Explain what you learned:** I can build a DataFrame from records and identify a missing field.

### M04C03. Look at the beginning and the end

The setup provides students(), a helper that returns a fresh copy of the 12-row fictional practice table. head(n) shows the first n rows and tail(n) shows the last n rows. These previews help you understand a file without printing the whole dataset. A preview alone does not prove every row is valid.

**Read:** Is students() a built-in pandas function?

Your explanation: _write here_.

**Predict:** Which student IDs appear in tail(2)?

Your explanation: _write here_.

**Explain:** Why should you avoid deciding that a whole dataset is clean after reading only head()?

Your explanation: _write here_.



In [ ]:
# Predict before running: Which student IDs appear in tail(2)?
df = students()
first = df.head(3)
last = df.tail(2)
print("First three rows:")
print(first)
print("Last two rows:")
print(last)

**Your turn:** Show the first two rows and last three rows. Print the full table's shape to confirm previews have not removed rows.

For the check, keep these result names: `df`, `first`, `last`.

**Hint:** Use head(2), tail(3), and df.shape.

**Watch for this:** head() without an argument defaults to five rows, or fewer when fewer exist.

In [ ]:
# Change this code to complete the task.
# Question: Show the first two rows and last three rows. Print the full table's shape to confirm previews have not removed rows.
df = students()
first = df.head(3)
last = df.tail(2)
print("First three rows:")
print(first)
print("Last two rows:")
print(last)

**Quick check:** What does df.head(3) do?

1. Returns the first three rows.
2. Permanently deletes all other rows.
3. Returns the first three columns.

**Explain what you learned:** I can preview a dataset while preserving its full contents.

### M04C04. Inspect names, types, and a first summary

Attributes such as shape and columns are stored information, so they do not use parentheses. Methods such as info() perform an action and do use parentheses. info() reports column names, non-missing counts, and dtypes; describe() returns summary statistics. Text dtype labels can differ across pandas versions, so focus on what the data represent.

**Read:** Why is it df.shape but df.info()?

Your explanation: _write here_.

**Predict:** What will count be for both Age and Score in this clean table?

Your explanation: _write here_.

**Explain:** Why is df.info() better than only looking at the first row?

Your explanation: _write here_.



In [ ]:
# Predict before running: What will count be for both Age and Score in this clean table?
df = students()
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
df.info()
summary = df[["Age", "Score"]].describe()
print(summary.round(2))

**Your turn:** Inspect dtypes, then create a describe() summary for Study_Hours and Attendance only. Round it to one decimal place.

For the check, keep these result names: `summary`.

**Hint:** Use df.dtypes and select two columns with a list inside the brackets.

**Watch for this:** Call df.info() directly. print(df.info()) also prints None because info() returns None.

In [ ]:
# Change this code to complete the task.
# Question: Inspect dtypes, then create a describe() summary for Study_Hours and Attendance only. Round it to one decimal place.
df = students()
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
df.info()
summary = df[["Age", "Score"]].describe()
print(summary.round(2))

**Quick check:** Which expression correctly accesses the table shape?

1. df.shape()
2. df[shape]
3. df.shape

**Explain what you learned:** I can inspect table structure before writing an analysis.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates